<a href="https://colab.research.google.com/github/Aditi0912dec/OLTVAR/blob/main/OLTVAR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<td>
   <a target="_blank" href="https://labelbox.com" ><img src="https://labelbox.com/blog/content/images/2021/02/logo-v4.svg" width=256/></a>
</td>


<td>
<a href="https://colab.research.google.com/github/Labelbox/labelbox-python/blob/master/examples/integrations/sam/meta_sam_labelbox_video.ipynb" target="_blank"><img
src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>
</td>

<td>
<a href="https://github.com/Labelbox/labelbox-python/blob/master/examples/integrations/sam/meta_sam_labelbox_video.ipynb" target="_blank"><img
src="https://img.shields.io/badge/GitHub-100000?logo=github&logoColor=white" alt="GitHub"></a>
</td>

# Setting the stage

First, we import and prepare the prerequisites to process the video.

### General dependencies

In [ ]:
!nvidia-smi

Tue May 21 09:50:36 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   59C    P8              10W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [ ]:
import os
#HOME = os.getcwd()
#print(HOME)

import sys
from google.colab.patches import cv2_imshow
import cv2
import PIL
from PIL import Image
import numpy as np
import uuid
import tempfile

from IPython import display
display.clear_output()
from IPython.display import display, Image
from io import BytesIO

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


download ucf50 dataset

In [ ]:
# Discard the output of this cell.
%%capture

# Downlaod the UCF50 Dataset
!wget --no-check-certificate https://www.crcv.ucf.edu/data/UCF50.rar

#Extract the Dataset
!unrar x UCF50.rar

In [ ]:
#os.mkdir('UCF50_trial')

In [ ]:
#CLASS_LIST=os.listdir('UCF50')


In [ ]:
# for action in CLASS_LIST:
#   path=os.path.join('UCF50_trial',action)
#   os.mkdir(path)

In [ ]:
#video_list=os.listdir('/content/UCF50/HorseRiding')

In [ ]:
# You can also use the Labelbox Client API to get specific videos or an entire
# dataset from your Catalog. Refer to these docs:
# https://labelbox-python.readthedocs.io/en/latest/#labelbox.client.Client.get_data_row

#VIDEO_PATH = "/content/drive/MyDrive/action_recognition/baseball.mp4"

#%cd {HOME}
#!wget -v {VIDEO_PATH}

### YOLOv8 dependencies

In [ ]:
# Dependencies for YOLOv8

!pip install ultralytics==8.0.20

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.2/261.2 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.0/289.0 kB 7.1 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manyl

In [ ]:
# Import packages

import ultralytics
ultralytics.checks()
from ultralytics import YOLO

Ultralytics YOLOv8.0.20 🚀 Python-3.10.12 torch-2.3.0+cu121 CPU
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 36.2/107.7 GB disk)


In [ ]:
# Instantiate YOLOv8 model

model = YOLO('yolov8n.pt')
#colors = np.random.randint(0, 256, size=(len(model.names), 3))
#colors = np.column_stack([np.random.randint(0, 256, size=(len(model.names), 3)), np.full((len(model.names), 1), 255)])
#model.names

# Specify which classes you care about. The rest of classes will be filtered out.
#chosen_class_ids = [0] # person # BALL

100%|██████████| 6.23M/6.23M [00:00<00:00, 42.8MB/s]



In [ ]:
model.names

{0: 'person',
 1: 'bicycle',
 2: 'car',
 3: 'motorcycle',
 4: 'airplane',
 5: 'bus',
 6: 'train',
 7: 'truck',
 8: 'boat',
 9: 'traffic light',
 10: 'fire hydrant',
 11: 'stop sign',
 12: 'parking meter',
 13: 'bench',
 14: 'bird',
 15: 'cat',
 16: 'dog',
 17: 'horse',
 18: 'sheep',
 19: 'cow',
 20: 'elephant',
 21: 'bear',
 22: 'zebra',
 23: 'giraffe',
 24: 'backpack',
 25: 'umbrella',
 26: 'handbag',
 27: 'tie',
 28: 'suitcase',
 29: 'frisbee',
 30: 'skis',
 31: 'snowboard',
 32: 'sports ball',
 33: 'kite',
 34: 'baseball bat',
 35: 'baseball glove',
 36: 'skateboard',
 37: 'surfboard',
 38: 'tennis racket',
 39: 'bottle',
 40: 'wine glass',
 41: 'cup',
 42: 'fork',
 43: 'knife',
 44: 'spoon',
 45: 'bowl',
 46: 'banana',
 47: 'apple',
 48: 'sandwich',
 49: 'orange',
 50: 'broccoli',
 51: 'carrot',
 52: 'hot dog',
 53: 'pizza',
 54: 'donut',
 55: 'cake',
 56: 'chair',
 57: 'couch',
 58: 'potted plant',
 59: 'bed',
 60: 'dining table',
 61: 'toilet',
 62: 'tv',
 63: 'laptop',
 64: 'mou

### SAM dependencies

In [ ]:
# Download SAM model SDK

#%cd {HOME}
!{sys.executable} -m pip install 'git+https://github.com/facebookresearch/segment-anything.git'

  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-_86wx12a
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-_86wx12a
  Resolved https://github.com/facebookresearch/segment-anything.git to commit 6fdee8f2727f4506cfbbe553e23b895e27956588
  Preparing metadata (setup.py) ... done
  Created wheel for segment-anything: filename=segment_anything-1.0-py3-none-any.whl size=36590 sha256=e280c7be17cdaf06203a58dc74f3729336f2ef87f2bd025d57cd59b8e1623f32
  Stored in directory: /tmp/pip-ephem-wheel-cache-q4hgnpbj/wheels/10/cf/59/9ccb2f0a1bcc81d4fbd0e501680b5d088d690c6cfbc02dc99d
Successfully built segment-anything


In [ ]:
!wget 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth'

--2024-05-30 16:35:21--  https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 18.173.166.31, 18.173.166.51, 18.173.166.48, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|18.173.166.31|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2564550879 (2.4G) [binary/octet-stream]
Saving to: ‘sam_vit_h_4b8939.pth’

sam_vit_h_4b8939.pt 100%[===================>]   2.39G  44.2MB/s    in 23s     

2024-05-30 16:35:44 (107 MB/s) - ‘sam_vit_h_4b8939.pth’ saved [2564550879/2564550879]



In [ ]:
import torch
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
MODEL_TYPE = "vit_h"


In [ ]:
# Import packages

#import torch
import matplotlib.pyplot as plt
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor

In [ ]:
# Instantiate SAM model

#DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
sam = sam_model_registry["vit_h"](checkpoint="sam_vit_h_4b8939.pth").to(device=device)
mask_predictor = SamPredictor(sam)

### Helper functions

In [ ]:
# Cast color to ints
#def get_color(color):
 # return (int(color[0]), int(color[1]), int(color[2]))
############# white masking ############
def get_color():
  return (int(255), int(255), int(255))

# Get video dimensions
def get_video_dimensions(input_cap):
  width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
  height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
  return height, width

# Get output video writer with same dimensions and fps as input video
def get_output_video_writer(input_cap, output_path):
  # Get the video's properties (width, height, FPS)
  width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
  height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
  fps = int(cap.get(cv2.CAP_PROP_FPS))

  # Define the output video file
  output_codec = cv2.VideoWriter_fourcc(*"mp4v")  # MP4 codec
  output_video = cv2.VideoWriter(output_path, output_codec, fps, (width, height))

  return output_video

# Visualize a video frame with bounding boxes, classes and confidence scores
def visualize_detections(frame, boxes, conf_thresholds, class_ids):
    frame_copy = np.copy(frame)
    for idx in range(len(boxes)):
        class_id = int(class_ids[idx])
        conf = float(conf_thresholds[idx])
        x1, y1, x2, y2 = int(boxes[idx][0]), int(boxes[idx][1]), int(boxes[idx][2]), int(boxes[idx][3])
        #color = colors[class_id]
        label = f"{model.names[class_id]}: {conf:.2f}"
        cv2.rectangle(frame_copy, (x1, y1), (x2, y2), get_color(), 2)
        cv2.putText(frame_copy, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, get_color(), 2)
    return frame_copy

def add_color_to_mask(mask, color):
  next_mask = mask.astype(np.uint8)
  next_mask = np.expand_dims(next_mask, 0).repeat(3, axis=0)
  next_mask = np.moveaxis(next_mask, 0, -1)
  #next_mask *= 255
  return next_mask * color

# Merge masks into a single, multi-colored mask
# def merge_masks_colored(masks, class_ids):
# filtered_class_ids = []
# filtered_masks = []
# for idx, cid in enumerate(class_ids):
#   if int(cid) in chosen_class_ids:
#     filtered_class_ids.append(cid)
#     filtered_masks.append(masks[idx])

# merged_with_colors = add_color_to_mask(filtered_masks[0][0], get_color(colors[int(filtered_class_ids[0])])).astype(np.uint8)

# if len(filtered_masks) == 1:
#   return merged_with_colors

# for i in range(1, len(filtered_masks)):
#   curr_mask_with_colors = add_color_to_mask(filtered_masks[i][0], get_color(colors[int(filtered_class_ids[i])]))
#   merged_with_colors = np.bitwise_or(merged_with_colors, curr_mask_with_colors)

# return merged_with_colors.astype(np.uint8)

  ############################## for layer wise############

def merge_masks_colored(masks, class_ids,frame_val):
    merged_with_colors_list=[]

    for k in range(len(chosen_class_ids)):
      print("chosen class id :"+str(chosen_class_ids[k]))
      print(f"the value of k : {k}")
      filtered_class_ids = []
      filtered_masks = []
      for idx, cid in enumerate(class_ids):
        if int(cid)==chosen_class_ids[k]:
          filtered_class_ids.append(cid)
          filtered_masks.append(masks[idx])

      if len(filtered_masks)==0: # the object of iterest is not in that frame then put a black canvas in place
        print(f"in filter mask 0")
        blank_frame=np.zeros((frame_val[0],frame_val[1],frame_val[2]))
        blank_frame=cv2.cvtColor(blank_frame.astype(np.uint8), cv2.COLOR_BGR2GRAY)
        merged_with_colors_list.append(blank_frame)
      else:


        merged_with_colors = add_color_to_mask(filtered_masks[0][0], get_color())

        if len(filtered_masks) == 1:
          print(f"in filter mask 1")
          merged_with_colors_list.append(cv2.cvtColor(merged_with_colors.astype(np.uint8),cv2.COLOR_BGR2GRAY))
        else:
          print(f"in filter mask >1")
          for i in range(1, len(filtered_masks)):
            curr_mask_with_colors = add_color_to_mask(filtered_masks[i][0], get_color())
            merged_with_colors = np.bitwise_or(merged_with_colors, curr_mask_with_colors)
          merged_with_colors_list.append(cv2.cvtColor(merged_with_colors.astype(np.uint8),cv2.COLOR_BGR2GRAY))

    return merged_with_colors_list




### setup

In [ ]:

# Specify the height and width to which each video frame will be resized in our dataset.
IMAGE_HEIGHT , IMAGE_WIDTH = 64, 64

# Specify the number of frames of a video that will be fed to the model as one sequence.


# Specify the directory containing the UCF50 dataset.
DATASET_DIR = "UCF50"



In [ ]:
# Specify the list containing the names of the classes used for training. Feel free to choose any set of classes.
CLASSES_LIST = ["HorseRace"]

In [ ]:
import shutil



In [ ]:
import os
OUTPUT_DIR="UCF50_reduce"
os.mkdir(OUTPUT_DIR)

In [ ]:
def create_video_from_frames(frames_folder, output_video_path, frame_rate=30):
    # Get a list of all the frame files in the folder
    frame_files = [f for f in os.listdir(frames_folder) if os.path.isfile(os.path.join(frames_folder, f))]

    # Sort the frame files by name
    frame_files.sort()

    # Read the first frame to get the width and height
    first_frame = cv2.imread(os.path.join(frames_folder, frame_files[0]))
    height, width, layers = first_frame.shape

    # Define the codec and create a VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for .mp4 files
    video = cv2.VideoWriter(output_video_path, fourcc, frame_rate, (width, height))

    # Write each frame to the video
    for frame_file in frame_files:
        frame_path = os.path.join(frames_folder, frame_file)
        frame = cv2.imread(frame_path)
        video.write(frame)

    # Release the VideoWriter object
    video.release()

In [ ]:
def create_dataset():
    '''
    This function will extract the data of the selected classes and create the required dataset.
    Returns:
        features:          A list containing the extracted frames of the videos.
        labels:            A list containing the indexes of the classes associated with the videos.
        video_files_paths: A list containing the paths of the videos in the disk.
    '''

    # Declared Empty Lists to store the features, labels and video file path values.
    features = []
    labels = []
    video_files_paths = []

    # Iterating through all the classes mentioned in the classes list
    for class_index, class_name in enumerate(CLASSES_LIST):
        count=0
        # Display the name of the class whose data is being extracted.
        print(f'Extracting Data of Class: {class_name}')

        # Get the list of video files present in the specific class name directory.
        files_list = os.listdir(os.path.join(DATASET_DIR, class_name))

        # Iterate through all the files present in the files list.
        for file_name in files_list:

            # Get the complete video path.
            video_file_path = os.path.join(DATASET_DIR, class_name, file_name)
            !python candidate_frames_folder.py --input_videos video_file_path  --output_folder_video_image candidate_frames_and_their_cluster_folder --output_folder_video_final_image final_images
           # frames_folder = "/content/v_Biking_g01_c01/final_images"
            frame_folder=video_file_path+"/final_images"
           # print(video_file_path)
            output_dir_path=os.path.join(OUTPUT_DIR,class_name)

            output_file_path= os.path.join(output_dir_path, f"output_{file_name.split('.')[0]}.mp4")
            os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
            print(output_file_path)
            # Extract the frames of the video file.
            create_video_from_frames(frames_folder,output_file_path, frame_rate=30)
            count=count+1
            if count==2:
              break


            # Check if the extracted frames are equal to the SEQUENCE_LENGTH specified above.
            # So ignore the vides having frames less than the SEQUENCE_LENGTH.


In [ ]:
create_dataset()

Extracting Data of Class: HorseRace
UCF50_reduce/HorseRace/output_v_HorseRace_g10_c04.mp4
UCF50_reduce/HorseRace/output_v_HorseRace_g11_c07.mp4


In [ ]:
key_list = list(model.names.keys())
val_list = list(model.names.values())
chosen_class_ids=list()
obj_of_interest=["person","horse"]
for name in obj_of_interest:
  pos=val_list.index(name)
  chosen_class_ids.append(key_list[pos])
print(f"chosen class index :{ chosen_class_ids}")

In [ ]:
OLT_dataset=list()
frame_list=list()
#trainind_data_label=list()
for class_index, class_name in enumerate(CLASSES_LIST):

        # Display the name of the class whose data is being extracted.
    print(f'Extracting Data of Class: {class_name}')

          # Get the list of video files present in the specific class name directory.
    #files_list = os.listdir(os.path.join(DATASET_DIR, class_name))

    files_list = os.listdir(os.path.join(DATASET_DIR, class_name))
    count=1
    for file_name in files_list:
        layer_sample_per_video=list()
        print(file_name)
        video_file_path= os.path.join("/content",OUTPUT_DIR, class_name,f"output_{file_name.split('.')[0]}.mp4")
        print(video_file_path)
        #/content/UCF50_reduce/Biking/output_v_Biking_g02_c03.mp4
        cap = cv2.VideoCapture(video_file_path)
        print(f"cap status: { cap.isOpened()}")
       # print(video_file_path)
       # output_video_boxes=get_output_video_writer(cap,os.makedirs(os.path.dirname(os.path.join(Boxes,class_name,f"{file_name.split('.')[0]}_boxes.mp4")),exist_ok=True))
       # output_video_masks=get_output_video_writer(cap,os.makedirs(os.path.dirname(os.path.join(Masks,class_name,f"{file_name.split('.')[0]}_masks.mp4")),exist_ok=True))
       # mask_frames = []

       # count=count+1
       # print(count)
       # if count>90:
        #  break
        # Loop through the frames of the video
        frame_num = 1
        while cap.isOpened():

          if frame_num % 30 == 0 or frame_num == 1:
            print("Processing frames", frame_num, "-", frame_num+29)
          print(frame_num)
          ret, frame = cap.read()


          if not ret:
              break

          # Run frame through YOLOv8 to get detections
          detections = model.predict(frame, conf=0.10) # frame is a numpy array

          # Write detections to output video
          frame_with_detections = visualize_detections(frame,
                                                        detections[0].boxes.cpu().xyxy,
                                                        detections[0].boxes.cpu().conf,
                                                        detections[0].boxes.cpu().cls)
          #output_video_boxes.write(frame_with_detections)
          print(f"detected classes and conf for frame {frame_num}")
          for i in range(len(detections[0]))
          print((detections[0].boxes.cpu().cls[i],detections[0].boxes.cpu().conf[i]))

          # Run frame and detections through SAM to get masks
          transformed_boxes = mask_predictor.transform.apply_boxes_torch(detections[0].boxes.xyxy, list(get_video_dimensions(cap)))
          if len(transformed_boxes) == 0:
            print("No boxes found on frame", frame_num)
           # output_video_masks.write(frame)
            frame_num += 1
            continue
          mask_predictor.set_image(frame)
          masks, scores, logits = mask_predictor.predict_torch(
            boxes = transformed_boxes,
            multimask_output=False,
            point_coords=None,
            point_labels=None
          )
          masks = np.array(masks.cpu())
          if masks is None or len(masks) == 0:
            print("No masks found on frame", frame_num)
            #output_video_masks.write(frame)
            frame_num += 1
            continue
          #frame=np.dstack([frame, np.full((frame.shape[0], frame.shape[1]), 255, dtype=np.uint8)])
          merged_colored_mask = merge_masks_colored(masks, detections[0].boxes.cls,frame.shape)
          merged_layer_per_frame=list()
          for i in range(0,len(merged_colored_mask)):
            merged_layer_per_frame.append(merged_colored_mask[i])
          merged_layer_per_frame_tupple=tuple(merged_layer_per_frame)
          layer_sample_per_video.append(merged_layer_per_frame_tupple)

          #local_layer_list.append((merged_colored_mask[0],merged_colored_mask[1]))



          frame_num += 1
          #break

          # For the purposes of this demo, only look at the first 90 frames
          if frame_num > 90:
            break
        OLT_dataset.append((layer_sample_per_video,class_name))
        #trainind_data_label.append(class_name)
        cap.release()
        cv2.destroyAllWindows()
        break





Extracting Data of Class: HorseRace
v_HorseRace_g10_c04.avi
/content/UCF50_reduce/HorseRace/output_v_HorseRace_g10_c04.mp4
cap status: True
Processing frames 1 - 30
1
chosen class id :0
the value of k : 0
in filter mask 0
chosen class id :17
the value of k : 1
in filter mask >1
2
chosen class id :0
the value of k : 0
in filter mask >1
chosen class id :17
the value of k : 1
in filter mask >1
3
chosen class id :0
the value of k : 0
in filter mask 0
chosen class id :17
the value of k : 1
in filter mask >1
4
chosen class id :0
the value of k : 0
in filter mask 0
chosen class id :17
the value of k : 1
in filter mask >1
5
chosen class id :0
the value of k : 0
in filter mask 0
chosen class id :17
the value of k : 1
in filter mask >1
6
chosen class id :0
the value of k : 0
in filter mask 0
chosen class id :17
the value of k : 1
in filter mask >1
7
chosen class id :0
the value of k : 0
in filter mask >1
chosen class id :17
the value of k : 1
in filter mask >1
8
chosen class id :0
the value of k

In [ ]:
merged_colored_mask = [[1, 2], [1, 3], [1, 5]]  # Your list of elements

a=list()

# Iterate to add elements to the list
for k in [0,1]:
  layer_dataset = []
  for element in merged_colored_mask:
      layer_dataset.append(element)

  # Convert the list to a tuple
  dataset = tuple(layer_dataset)

  # Print the result
  print(dataset)

  a.append(dataset)

([1, 2], [1, 3], [1, 5])
([1, 2], [1, 3], [1, 5])


In [ ]:
print(a)

[([1, 2], [1, 3], [1, 5]), ([1, 2], [1, 3], [1, 5])]


In [ ]:
layer_dataset

[([1, 2], [1, 3]), ([1, 3], [1, 5])]